In [142]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model
import importlib
import mod02_build_bot_predictor
importlib.reload(mod02_build_bot_predictor)
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [143]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [144]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [145]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [146]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [147]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [148]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [149]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 184,
 'tn': 2623,
 'fp': 14,
 'fn': 179,
 'misclassification_rate': 0.06433333333333334,
 'false_positive_rate': 0.005309063329541145,
 'false_negative_rate': 0.4931129476584022}

### Check results on the test set (new data not yet seen by the model)

In [150]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 32,
 'tn': 852,
 'fp': 22,
 'fn': 94,
 'misclassification_rate': 0.116,
 'false_positive_rate': 0.02517162471395881,
 'false_negative_rate': 0.746031746031746}

# Discussion Questions

In [152]:
y_test.value_counts()

is_bot
0    874
1    126
Name: count, dtype: int64

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

While the actual misclassification rate is quite low, suggesting a strong model, looking at the data distribution and the false pos/neg rates is more informative. 874 samples from the test set are human posts, while 126 are from bots. The training set has the same distribution issue, with 363 bot posts and 2637 human posts. The model is very likely just guessing human because of the frequency of human posts compared to bot posts, not because of any novel differences it found between the two kinds of posts. This also explains the very high false negative rate vs. the low false positive rate.

### What are potential ramifications of false positives from the model?

False positives would imply that the model is classifying human-made posts as being authored by bots. This removes credence from those posts, and means that actual human text could be marked as spam and removed.

### What are potential ramifications of false negatives from the model?

False negatives imply that the model is classifying bot-made posts as being authored by humans. This means that actual humans could misconstrue the text as being a genuine opinion, and may end up basing their own opinions on something that is completely artificial. 